# Context-Aware Chatbot Using LangChain or RAG

## Objective:
Build a conversational chatbot that can remember context and retrieve external information during conversations.

## Setup: Install necessary libraries

In [ ]:
%%capture
pip install langchain==0.1.16 sentence-transformers==2.7.0 faiss-cpu==1.8.0

## Load Data from Webpages

First, we'll define a list of URLs and use `WebBaseLoader` to load the content from these pages. Remember to replace the example URLs with your actual knowledge base pages.

In [ ]:
from langchain.schema import Document
from langchain_community.document_loaders import WebBaseLoader

# Define the URLs for your knowledge base
# Replace these example URLs with your actual web pages
urls = [
    "https://www.ibm.com/topics/large-language-models",
    "https://aws.amazon.com/what-is/rag/",
    "https://www.techtarget.com/whatis/definition/Generative-AI"
]

# Load documents from the URLs
loader = WebBaseLoader(urls)
documents = loader.load()

print(f"Loaded {len(documents)} documents.")

## Split Documents into Chunks

Large documents need to be split into smaller, more manageable chunks for effective embedding and retrieval. `RecursiveCharacterTextSplitter` is a good choice for this.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks.")

## Create Embeddings and Vector Store

Now, we'll create embeddings for each text chunk using a `HuggingFaceEmbeddings` model and store them in a `FAISS` vector store. This allows us to efficiently search for relevant document chunks based on semantic similarity.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the embeddings model
# You can choose different models (e.g., 'all-MiniLM-L6-v2')
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the document chunks and embeddings
vector_store = FAISS.from_documents(texts, embeddings)

print("Vector store created successfully!")

With the vector store ready, we now have a searchable index of your knowledge base. The next step would typically involve setting up the LLM, the retriever, and the conversational chain.

## Initialize the Language Model (LLM)

We'll use a `ChatGoogleGenerativeAI` model from LangChain as our LLM for generating responses. This requires your Google API key to be set up as we did at the beginning of this session.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# Ensure your GOOGLE_API_KEY is set up in Colab Secrets
llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key=userdata.get("GOOGLE_API_KEY"))

print("LLM initialized successfully!")

## Set up the Retriever

The retriever's job is to fetch the most relevant document chunks from our `vector_store` based on the user's query. This is where the RAG aspect comes in, ensuring our chatbot's responses are grounded in your custom corpus.

In [ ]:
retriever = vector_store.as_retriever()

print("Retriever set up from FAISS vector store!")

## Implement Context Memory for Conversational History

To make the chatbot context-aware, we need to store the conversation history. `ConversationBufferMemory` from LangChain helps maintain the flow of the dialogue.

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

print("Conversational memory initialized!")

## Create the Conversational Retrieval Chain

Finally, we'll combine the LLM, retriever, and memory into a single conversational retrieval chain. This chain will take user input, retrieve relevant documents, and generate a coherent response while maintaining conversational context.

In [ ]:
from langchain.chains import ConversationalRetrievalChain

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

print("Conversational Retrieval Chain created!")

Now your chatbot is ready to answer questions based on your provided webpages and maintain conversation history! You can test it by asking a query related to the content of your `urls`.

In [ ]:
# Example usage:
query = "What is RAG?"
result = qa_chain.invoke({"question": query})
print(f"Chatbot: {result['answer']}")

## Deploy the Chatbot with Streamlit

To deploy the chatbot as a web application, we'll use Streamlit. First, you'll need to install Streamlit. Then, we'll create a Python script that uses the `qa_chain` we've already built and serves it with a Streamlit UI.

In [ ]:
%%capture
pip install streamlit

Now, let's create the Streamlit application. You can save the following code as a Python file (e.g., `app.py`) and run it. For Colab, we'll write it to a file and then use `streamlit run`.

In [ ]:
%%writefile app.py

import streamlit as st
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from google.colab import userdata

# Securely get the Google API Key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# --- Data Loading, Splitting, and Vector Store (replicate previous steps) ---
@st.cache_resource
def get_vector_store():
    urls = [
        "https://www.ibm.com/topics/large-language-models",
        "https://aws.amazon.com/what-is/rag/",
        "https://www.techtarget.com/whatis/definition/Generative-AI"
    ]
    loader = WebBaseLoader(urls)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )
    texts = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(texts, embeddings)
    return vector_store

vector_store = get_vector_store()
retriever = vector_store.as_retriever()

# --- LLM and Chain Setup (replicate previous steps) ---
@st.cache_resource
def get_conversational_chain():
    llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key=GOOGLE_API_KEY)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory
    )
    return qa_chain

qa_chain = get_conversational_chain()

# --- Streamlit UI ---
st.set_page_config(page_title="Context-Aware Chatbot")
st.title("Context-Aware Chatbot")
st.write("Ask me anything about LLMs, RAG, and Generative AI!")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# React to user input
if prompt := st.chat_input("What would you like to know?"):
    # Display user message in chat message container
    st.chat_message("user").markdown(prompt)
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.spinner("Thinking..."):
        response = qa_chain.invoke({"question": prompt})
        answer = response["answer"]

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        st.markdown(answer)
    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": answer})


To run this Streamlit app in Colab, you'll need `ngrok` to expose the local server. Make sure to get an authtoken from ngrok (it's free) and replace `YOUR_NGROK_AUTHTOKEN` below.

In [ ]:
%%capture
!pip install pyngrok==10.0.0
from pyngrok import ngrok

# Replace with your ngrok authtoken
# You can get one for free from https://ngrok.com/
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

# If you have your authtoken in Colab secrets as 'NGROK_AUTH_TOKEN'
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Terminate any previous ngrok tunnels
ngrok.kill()

# Start a ngrok tunnel for Streamlit (port 8501)
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

In [ ]:
!streamlit run app.py &>/dev/null

## Summary and Key Insights

### Summary

In this project, we successfully built a context-aware chatbot using LangChain and Retrieval-Augmented Generation (RAG). We started by setting up the necessary libraries, including `langchain`, `sentence-transformers`, and `faiss-cpu`. We then loaded a custom corpus from specified webpages, split the content into manageable chunks, and created a FAISS vector store with embeddings generated by `HuggingFaceEmbeddings`.

Next, we integrated a Large Language Model (`ChatGoogleGenerativeAI`), configured a retriever to fetch relevant information from our vector store, and implemented conversational memory to maintain dialogue context. These components were combined into a `ConversationalRetrievalChain`, enabling the chatbot to answer questions based on the provided web content while remembering previous interactions. Finally, we deployed this chatbot as a web application using Streamlit, exposed via `ngrok`, allowing for interactive testing.

### Key Insights

*   **RAG's Effectiveness:** The RAG approach significantly enhances the chatbot's ability to provide accurate and relevant answers by grounding the LLM's responses in a specific knowledge base, reducing hallucinations.
*   **LangChain's Modularity:** LangChain's modular design made it straightforward to combine different components (LLM, retriever, memory, chain) to build a sophisticated conversational agent.
*   **Importance of Chunking and Embeddings:** Proper text chunking and the selection of an effective embedding model are crucial for the performance of the retrieval system, directly impacting the relevance of retrieved information.
*   **Conversational Memory:** Integrating conversational memory is vital for a natural user experience, allowing the chatbot to understand and respond within the context of an ongoing dialogue.
*   **Streamlit for Rapid Deployment:** Streamlit proved to be an excellent tool for quickly deploying a user-friendly interface for the chatbot, making it accessible for testing and demonstration.